# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all record sets contained in the dataset, along with their `@id` and the fields they contain (with `@id` and field names).


In [ ]:
# List all record sets and their field IDs
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in metadata!\nTrying to load from distribution...")
    # Try to access distribution entries
    for distribution in getattr(metadata, 'distribution', []):
        print(f" - Distribution @id: {getattr(distribution, '@id', None)}")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                print(f"    {field['@id']}: {field.get('name', '')}")
        if 'column' in rs:
            print("  Columns:")
            for col in rs['column']:
                print(f"    {col['@id']}: {col.get('name', '')}")

For convenience, below is an example listing record sets and a first record from a record set. If the dataset has record sets by `@id`, replace with the correct ones below, as found above.

In [ ]:
# List records for each record set by @id
list_record_set_ids = [rs['@id'] for rs in dataset.record_sets()] if dataset.record_sets() else []
for record_set_id in list_record_set_ids:
    print(f'First record for Record Set {record_set_id}:')
    records_iter = dataset.records(record_set=record_set_id)
    for idx, record in enumerate(records_iter):
        print(record)
        if idx >= 0:
            break


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set found by @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets()] if dataset.record_sets() else []
dataframes = {}

for record_set_id in record_set_ids:
    try:
        print(f"Loading records from record set {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records for {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# Display the columns and first few records for the first record set
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Columns in first record set ({first_rs}):", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Specify a numeric field to analyze by @id or name (Adjust as needed based on the actual field names in the record set)
selected_record_set = list(dataframes.keys())[0] if dataframes else None
numeric_field = None
group_field = None

if selected_record_set:
    df_ex = dataframes[selected_record_set]
    # Try to find a suitable numeric field (e.g., MW, coverage, peptide_count, etc.)
    # Print sample columns to assist
    print("Columns available for EDA:", df_ex.columns.tolist())
    for col in df_ex.columns:
        if df_ex[col].dtype in [np.float64, np.int64, float, int]:
            numeric_field = col
            break
    # Otherwise, try to coerce suitable columns
    if not numeric_field:
        for col in df_ex.columns:
            # Try casting
            try:
                df_ex[col] = pd.to_numeric(df_ex[col], errors='ignore')
                if df_ex[col].dtype in [np.float64, np.int64, float, int]:
                    numeric_field = col
                    break
            except Exception:
                continue
    # Choose a possible group field (category/text)
    for col in df_ex.columns:
        if df_ex[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break

    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        threshold = df_ex[numeric_field].mean() if df_ex[numeric_field].dtype in [np.float64, np.int64, float, int] else 0
        filtered_df = df_ex[df_ex[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (mean):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of numeric field
if selected_record_set and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=dataframes[selected_record_set], x=numeric_field, kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, boxplot
    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[selected_record_set])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the mlcroissant library. We loaded the metadata, enumerated available record sets and fields by `@id`, extracted data into DataFrames by record set, and performed basic exploratory data analysis.

- We identified key numeric and group fields and performed filtering and z-score normalization.
- Visualizations illustrated the distributions and group-wise differences, if present in the provided data.

This workflow demonstrates how, using the `mlcroissant` library, researchers can efficiently discover, parse, and analyze FAIR datasets following the Croissant schema. For further analysis or machine learning workflows, adapt the cell logic, referencing fields and record sets by their `@id` throughout.